[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/19_Context_Module.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 上下文模块（Context Module）—— 实战指南

Semantica 的 `context` 模块是让智能体"有状态"的那一层。LLM 本身没有记性——每次对话都是初次见面；这一层给智能体装上海马体。它组合了：

- **记忆**（短期 + 长期），经 `AgentMemory`
- **图上下文**，经 `ContextGraph`
- **混合检索**（向量 + 记忆 + 图），经 `ContextRetriever`
- **高层易用入口**，经 `AgentContext`（推荐入口）
- **实体链接**，经 `EntityLinker`
- **可扩展与配置**，经 `registry` 和 `config`

本 notebook 用小而可运行的例子，导入都限定在各自 cell 内。

### 先看主线

九节三段——**会用 → 会连 → 会拆**：

**会用（第 1–4 节）—— 记起来**

备好向量底座（长期记忆的家）、用 `AgentContext` 统一入口把记忆/向量/图接线、存取几条记忆试试、再学导出与持久化。

**会连（第 5–7 节）—— 连成图**

文档入库抽实体建图（GraphRAG 的地基）、在图上查询与扩展邻居、用 `EntityLinker` 给实体定稳定 URI 并归一。

**会拆（第 8–9 节）—— 拆开装**

高层接口不够用时直接接线低层积木（`AgentMemory` + `ContextRetriever`）；再把自己的方法注册进 `registry`。

一句话：记忆让智能体记得住，图让记忆连得起，检索让两者用得上。

In [ ]:
!pip install -q semantica

## 1) 向量库（长期记忆的底座）

`VectorStore` 能通过内置嵌入器生成向量。环境里没有嵌入器时，它会退回随机向量——API 保持可用，方便演示。

In [ ]:
from semantica.vector_store import VectorStore

vs = VectorStore(backend="inmemory", dimension=384)

if getattr(vs, "embedder", None) and hasattr(vs.embedder, "set_text_model"):
    vs.embedder.set_text_model(method="fastembed", model_name="BAAI/bge-small-en-v1.5")

vs.backend, vs.dimension

## 2) `AgentContext` 快速上手（推荐）

`AgentContext` 是把记忆、向量库和图接在一起的友好接口。传入 `ContextGraph` 后，系统就能做 GraphRAG 风格的检索。

In [ ]:
from semantica.context import AgentContext, ContextGraph

kg = ContextGraph()
context = AgentContext(vector_store=vs, knowledge_graph=kg)

context.config

## 3) 存取记忆

单个字符串就是一条记忆项。可以通过元数据友好的参数挂上 `conversation_id` 和 `user_id`。

In [ ]:
memory_id = context.store(
    "User prefers short answers about Python.",
    conversation_id="conv_1",
    user_id="user_1",
    metadata={"type": "preference"},
)

context.get_memory(memory_id)

In [ ]:
context.store(
    "User is working on Semantica context module examples.",
    conversation_id="conv_1",
    user_id="user_1",
    metadata={"type": "note"},
)

context.retrieve("Python answers", max_results=3)

In [ ]:
context.conversation("conv_1", max_items=10)

## 4) 导出、保存、加载

`AgentContext` 自带简单的持久化工具。这个例子用临时目录。

In [ ]:
export_json = context.export(conversation_id="conv_1", format="json")
export_json[:300]

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as d:
    context.save(d)
    context.load(d)

context.conversation_summary("conv_1")

## 5) 存文档并构建上下文图

存列表时，`AgentContext.store(...)` 把它当文档处理。为了让本 notebook 轻量且确定，这里直接传入预先抽好的实体和关系。

In [ ]:
documents = [
    {
        "id": "doc_1",
        "content": "Python is used for machine learning.",
        "metadata": {"source": "docs"},
        "entities": [
            {"id": "e_python", "text": "Python", "type": "PROGRAMMING_LANGUAGE"},
            {"id": "e_ml", "text": "Machine Learning", "type": "CONCEPT"},
        ],
        "relationships": [
            {
                "source_id": "e_python",
                "target_id": "e_ml",
                "type": "used_for",
                "confidence": 0.9,
            }
        ],
    },
    {
        "id": "doc_2",
        "content": "PyTorch is a machine learning framework.",
        "metadata": {"source": "docs"},
        "entities": [
            {"id": "e_pytorch", "text": "PyTorch", "type": "FRAMEWORK"},
            {"id": "e_ml", "text": "Machine Learning", "type": "CONCEPT"},
        ],
        "relationships": [
            {
                "source_id": "e_pytorch",
                "target_id": "e_ml",
                "type": "implements",
                "confidence": 0.95,
            }
        ],
    },
]

stats = context.store(
    documents,
    extract_entities=False,
    extract_relationships=False,
    link_entities=True,
)

stats

In [ ]:
kg.stats()

## 6) 用 `ContextGraph` 探索图

图支持关键词查询和邻居扩展。

In [ ]:
kg.query("machine learning")

In [ ]:
kg.get_neighbors("e_python", hops=2)

## 7) 用 `EntityLinker` 做实体链接

`EntityLinker` 分配稳定的 URI，还能跨来源链接相关或重复的实体——18 篇去重的"身份归一"在这里有了统一门牌。

In [ ]:
from semantica.context import EntityLinker

linker = EntityLinker(knowledge_graph={"entities": [{"id": "e_py", "text": "Python", "type": "PROGRAMMING_LANGUAGE"}]})

entities = [
    {"id": "e1", "text": "Python", "type": "PROGRAMMING_LANGUAGE"},
    {"id": "e2", "text": "PyTorch", "type": "FRAMEWORK"},
]

linked = linker.link("Python and PyTorch", entities=entities)
[(e.entity_id, e.uri, len(e.linked_entities)) for e in linked]

In [ ]:
linker.link_entities("e1", "e2", link_type="related_to", confidence=0.8)
linker.get_entity_links("e1")[:2]

In [ ]:
linker.build_entity_web()["statistics"]

## 8) 低层积木：`AgentMemory` 与 `ContextRetriever`

想要比 `AgentContext` 更细的控制？可以直接接线各个部件。

In [ ]:
from semantica.context import AgentMemory, ContextRetriever

memory = AgentMemory(vector_store=vs, knowledge_graph=kg, retention_policy="unlimited")
memory.store("Python powers Semantica.", metadata={"type": "fact", "conversation_id": "conv_2"})

retriever = ContextRetriever(memory_store=memory, knowledge_graph=kg, vector_store=vs)
results = retriever.retrieve("Python Semantica", max_results=5)

[(r.content, r.source, round(r.score, 3)) for r in results]

## 9) methods、registry 与配置

`methods` 层提供便捷函数，`registry` 让你插入自己的实现，`config` 提供运行时配置。

### 回顾主线

九节走完：**会用**（向量底座、AgentContext 入口、存取与持久化）让智能体记得住；**会连**（文档建图、图上查询、实体链接）让记忆连得起；**会拆**（低层接线、注册自定义方法）让你不受高层接口的限。记忆 + 图 + 混合检索——这就是智能体的海马体。

In [ ]:
from semantica.context.config import context_config

context_config.set("retention_policy", "7_days")
context_config.get("retention_policy")

In [ ]:
from semantica.context.methods import build_context_graph
from semantica.context.registry import method_registry

def custom_graph_method(entities, relationships, conversations=None, **kwargs):
    return {
        "nodes": [],
        "edges": [],
        "statistics": {"node_count": 0, "edge_count": 0},
    }

method_registry.register("graph", "custom_demo", custom_graph_method)
method_registry.list_all("graph")

In [ ]:
build_context_graph(
    entities=[{"id": "e1", "text": "Python", "type": "PROGRAMMING_LANGUAGE"}],
    relationships=[{"source_id": "e1", "target_id": "e2", "type": "related_to"}],
    method="custom_demo",
)